# vdb_embedding_colab.ipynb — KOSIS 표 28만여 개를 벡터DB용으로 임베딩

배경: `agent/kosis/crawl_table_catalog.py`로 KOSIS 통계표 전체(약 28만7천개)의
ID/이름/기관코드를 크롤링해뒀다(`agent/kosis/crawl_output/tables.jsonl`). 카탈로그
매칭(3단계)에서 지금 쓰는 64개 수동 카탈로그만으로는 커버리지가 부족해서, 이 28만개를
의미 기반 검색(임베딩)으로 보조 후보를 찾는 데 쓰려고 한다.

이만한 양을 로컬(RAM 7.4GB)에서 임베딩하면 위험해서, 기존 리랭커/임베딩 작업과 같은
방식으로 코랩에서 배치 처리한다. `agent/kosis/prepare_vdb_export.py` +
`agent/kosis/enrich_org_names.py`가 만든 `data/vdb_pending.jsonl`(표 ID + "기관명
(연-월) 표이름" 형태로 보강된 텍스트)을 여기서 읽어서 임베딩하고, 결과(벡터 + ID)를
다시 로컬로 가져가서 적재한다.

2026-08-18: 벡터DB를 Chroma에서 Supabase(pgvector)로 옮겼다 — 적재 스크립트가
`build_vdb_index.py`(Chroma 버전)에서 Postgres/pgvector용으로 바뀌었다. 이 노트북
자체(임베딩 생성)는 저장소가 뭐든 동일하게 쓸 수 있다.

2026-08-19: 골든셋 실측(Recall@5 0%에 가까움)을 계기로 e5-small(384차원)에서
**Qwen3-Embedding-4B**로 교체한다 — 팀 자체 비교실험(64개 카탈로그 기준 top-1 70%,
1위)과 소규모 VDB 사전 검증(Recall@5 29.3%, e5-small의 0%보다 뚜렷하게 개선)에서 이미
효과가 확인된 모델이다. 다만 4B 파라미터 모델의 원래 출력 차원(2560)을 그대로 쓰면
28만7천 건 기준 HNSW 인덱스까지 합쳐 약 7.5GB로 추정돼 Supabase Pro(8GB)에 거의
꽉 차게 들어간다 — Matryoshka 표현 학습이 된 모델이라 **1024차원으로 잘라도**(`truncate_dim`)
의미 성능이 크게 안 떨어지므로, 1024차원으로 잘라서 쓴다(예상 용량 ~3GB, 여유 확보).

⚠️ 4B 모델은 e5-small(118M)보다 훨씬 커서 코랩 무료 T4로는 버거울 수 있다 —
Colab Pro+(GPU 우선 배정, 백그라운드 실행, 최대 24시간 세션) 사용을 권장한다.

⚠️ 쿼리 쪽(claim 문장)도 반드시 같은 모델·같은 truncate_dim(1024)로 임베딩해야
벡터 공간이 맞는다 — 이 모델은 로컬(RAM 7.4GB)에서 못 돌아가므로, claim 쿼리 임베딩도
이제 코랩(`notebooks/reranker_colab.ipynb`)에서 처리하도록 별도로 맞춰야 한다.

## 사용법
1. 로컬에서 `python -m agent.kosis.prepare_vdb_export` → `python -m agent.kosis.enrich_org_names` 순서로 실행 → `data/vdb_pending.jsonl` 생성
2. 그 파일을 구글 드라이브 "내 드라이브" 최상위에 업로드
3. VSCode에서 이 노트북을 코랩 커널에 연결 (우측 상단 커널 선택 → Colab, GPU 런타임 권장)
4. 아래 셀을 순서대로 실행
5. 마지막 셀이 끝나면 구글 드라이브에 저장된 결과 파일 2개를 로컬 `data/`로 받아서
   `python -m agent.kosis.build_vdb_index`로 Supabase(pgvector)에 적재

## 1. 연결 확인

In [ ]:
import platform
print("플랫폼:", platform.platform())
try:
    import google.colab  # noqa: F401
    print("코랩에서 실행 중")
except ImportError:
    print("코랩 아님 — 로컬에서 도는 중 (커널 선택이 안 된 상태일 수 있음)")

!cat /proc/meminfo 2>/dev/null | head -3 || echo "(Linux 환경 아님)"

## 2. 드라이브 마운트 + 데이터 로드

`data/vdb_pending.jsonl`을 구글 드라이브 "내 드라이브" 최상위에 미리 업로드해두세요.

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

SRC = "/content/drive/MyDrive/vdb_pending.jsonl"
pending_filename = "vdb_pending.jsonl"
shutil.copy(SRC, pending_filename)
print("복사됨:", pending_filename)

In [ ]:
import json

rows = []
with open(pending_filename, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"표 {len(rows)}건 로드됨")
print("샘플:", rows[0])

## 3. 임베딩 생성

**Qwen3-Embedding-4B**를 1024차원으로 잘라서(`truncate_dim=1024`) 쓴다. Qwen3-Embedding
계열은 e5 계열과 달리 문서(passage) 쪽엔 별도 프리픽스가 필요 없다 — 쿼리 쪽에만
instruction을 붙이는 게 공식 권장 사용법이다(쿼리 임베딩은 이 노트북이 아니라
`reranker_colab.ipynb`에서 처리).

`.env`의 `KOSIS_EMBEDDING_MODEL`은 64개 카탈로그 매칭용(e5-small 계열)과는 별개다 —
VDB(28만7천여 개)만 Qwen으로 바꾸고, 64개 카탈로그는 그대로 e5 계열을 쓴다(용도가
분리돼 있어 서로 벡터 공간을 맞출 필요 없음).

4B 모델이라 e5-small보다 다운로드/추론이 훨씬 오래 걸린다 — GPU 런타임 필수,
Colab Pro+ 권장. 배치 크기도 모델 크기에 맞춰 줄였다(128 -> 16).

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"
EMBED_DIM = 1024  # Matryoshka 절단 차원 (원래 2560 -> 1024, Supabase 8GB 여유 확보용)

embed_model = SentenceTransformer(EMBED_MODEL_NAME, truncate_dim=EMBED_DIM)
print(f"모델 로드 완료: {EMBED_MODEL_NAME} (truncate_dim={EMBED_DIM})")

In [ ]:
import numpy as np
import time

# Qwen3-Embedding은 문서(passage) 쪽에 프리픽스가 필요 없다 — 표 텍스트를 그대로 인코딩.
texts = [r["text"] for r in rows]

t0 = time.time()
embeddings = embed_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"임베딩 완료: {embeddings.shape}, {time.time() - t0:.1f}초 소요")

## 4. 결과 저장 (드라이브로)

벡터는 JSON이 아니라 numpy 바이너리(.npy)로 저장한다 — 28만7천 × 1024차원(Qwen3-Embedding-4B,
truncate_dim=1024)을 텍스트로 저장하면 용량이 훨씬 커지고 느리다. ID/기관코드/원문은
임베딩 배열과 같은 순서로 저장된 별도 jsonl로 남겨서, 나중에 로컬에서 인덱스(줄 번호)로
다시 짝지을 수 있게 한다.

`files.download()`는 이 VS Code<->코랩 연결에서 실제 다운로드가 안 뜨는 문제가 있어서
(리랭커 노트북 때와 동일), 마운트된 드라이브에 직접 저장하는 방식으로 우회한다.

In [ ]:
np.save("vdb_embeddings.npy", embeddings.astype(np.float32))

with open("vdb_metadata.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps({"tbl_id": r["tbl_id"], "org_id": r.get("org_id"), "text": r["text"]}, ensure_ascii=False) + "\n")

shutil.copy("vdb_embeddings.npy", "/content/drive/MyDrive/vdb_embeddings.npy")
shutil.copy("vdb_metadata.jsonl", "/content/drive/MyDrive/vdb_metadata.jsonl")

print("저장 완료: /content/drive/MyDrive/vdb_embeddings.npy, vdb_metadata.jsonl")
print("두 파일을 로컬 data/ 폴더로 받아서 build_vdb_index.py를 실행하세요.")